# Inspecting and debugging naming runs

Start with `prepare`: it exposes cluster evidence and initial prompts without
calling the naming provider. This notebook uses a deterministic local provider;
the documentation build does not execute cells or connectivity checks.


In [ ]:
import numpy as np
from toponymy import Toponymy, PrecomputedClusterer

objects = [
    "An orchard grows apples.",
    "The orchard harvest begins in autumn.",
    "A river flows toward the coast.",
    "Rain raises the river level.",
]
embedding_vectors = np.array([[1.0, 0.0], [0.9, 0.1], [0.0, 1.0], [0.1, 0.9]])
label_layers = [np.array([10, 10, 42, 42]), np.array([7, 7, 7, 7])]


In [ ]:
import json

class LocalNamer:
    """Scripted responses at the naming boundary, with no service calls."""
    def __init__(self):
        self.calls = 0

    def generate_topic_name(self, prompt, *, response_parser):
        self.calls += 1
        response = {"topic_name": f"Demo topic {self.calls}", "topic_specificity": 0.8}
        return response_parser(json.dumps(response))

    def generate_topic_cluster_names(self, prompt, old_names, *, response_parser):
        self.calls += 1
        response = {
            "new_topic_name_mapping": {
                str(i): f"{name} ({i})" for i, name in enumerate(old_names, 1)
            },
            "topic_specificities": [0.8] * len(old_names),
        }
        return response_parser(json.dumps(response))


In [ ]:
namer = LocalNamer()
pipeline = Toponymy(
    namer,
    clusterer=PrecomputedClusterer(label_layers),
    object_description="notes",
    corpus_description="four short notes",
)
pipeline.prepare(objects, embedding_vectors)
assert namer.calls == 0
for key, topic in pipeline.topics_.items():
    print(key, topic.members, topic.prompt)


## Audit views

Audit helpers consume the fitted pipeline or its `TopicModel`. They read topic
state, not naming fields on cluster layers. These views work before naming to
inspect evidence, and after naming to compare it with results.


In [ ]:
from toponymy.audit import create_audit_df, create_layer_summary_df

print(create_layer_summary_df(pipeline))
print(create_audit_df(pipeline))
pipeline.name_topics()
print(pipeline.request_counts_)


## Provider callbacks

For an explicitly configured real provider, a callback can inspect transport
events. `BasicDebugLogger` appends JSONL locally; logs can contain prompt text
and raw responses, so select an appropriate destination for your corpus.

```python
from toponymy.debug_logging import BasicDebugLogger
from toponymy.llm_wrappers import LiteLLMNamer

namer = LiteLLMNamer(
    model=your_provider_model,
    api_key=your_api_key,
    callback=BasicDebugLogger("llm_debug.jsonl"),
)
```

Constructing a wrapper is separate from calling it. Connectivity checks can
send requests and are not needed to render prompts. Callback failures must not
hide the original provider result or error.

## Failures and continuation

Schema requirements, invalid input, authentication errors and unexpected
programming failures are reported explicitly. Transient retries are bounded;
invalid output does not silently become an empty topic name. Async cancellation
propagates to the caller, and batch results retain input ordering.

After a naming failure, inspect existing `topics_` and the exception. Successful
names remain visible in the prepared run. Once the underlying problem is fixed,
continuing `name_topics()` can complete that run; calling `fit()` instead starts
new data-dependent state. Upper-layer prompts may be refreshed when named
child-topic evidence becomes available.
